In [1]:
!pip install --upgrade jax jaxlib flax

import jax
# jax.config.update('jax_num_cpu_devices', 8)
import jax.numpy as jnp
import numpy as np

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 MB 82.9 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.9.0.1
    Uninstalling jaxlib-0.9.0.1:
      Successfully uninstalled jaxlib-0.9.0.1
  Attempting uninstall: jax
    Found existing installation: jax 0.9.0.1
    Uninstalling jax-0.9.0.1:
      Successfully uninstalled jax-0.9.0.1
  Attempting uninstall: optax
    Found existing installation: optax 0.2.7
    Uninstalling optax-0.2.7:
      Successfully uninstalled optax-0.2.7
  Attempting uninstall: flax
    Found existing installation: flax 0.12.3
    Uninstalling flax-0.12.3:
      Successfully uninstalled flax-0.12.3

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


In [2]:
!pip install treescope

import treescope
treescope.basic_interactive_setup(autovisualize_arrays=True)


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
jax.devices()

E0000 00:00:1773413325.154689      12 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: ===
learning/45eac/tfrc/runtime/common_lib.cc:238


[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0),
 TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0),
 TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0),
 TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0),
 TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0),
 TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0),
 TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0),
 TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]

In [4]:
!pip install tensorboard tensorboard-plugin-profile

/usr/local/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.9/23.9 MB 119.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 85.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.34.0rc2
    Uninstalling protobuf-7.34.0rc2:
      Successfully uninstalled protobuf-7.34.0rc2
  Attempting uninstall: googleapis-common-protos
    Found existing installation: googleapis-common-protos 1.56.1
    Uninstalling googleapis-common-protos-1.56.1:
      Successfully uninstalled googleapis-common-protos-1.56.1

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
with jax.profiler.trace("/kaggle/working/"):
  key = jax.random.key(0)
  x = jax.random.normal(key, (32, 1024, 8192)).astype(jnp.bfloat16)
  W = jax.random.normal(key, (8192, 8192 * 4)).astype(jnp.bfloat16)
  y = x @ W
  y.block_until_ready()

In [6]:
%load_ext tensorboard

In [7]:
# %tensorboard --logdir=/tmp/tensorboard

# Play around with stuff!

In [8]:
import jax
import jax.numpy as jnp

import numpy as np

import treescope
treescope.basic_interactive_setup(autovisualize_arrays=True)

from jax.sharding import PartitionSpec as P

# Can also do this to "mock" 1024 devices for the sake of parallelism.
# jax.config.update('jax_num_cpu_devices', 1024)

In [9]:
def forward(x: jax.Array,       # [B, T],
            embed: jax.Array,   # [V, D],
            w1: jax.Array,      # [D, F],
            w2: jax.Array       # [F, D]
    ) -> jax.Array:

    # Embed input tokens [B, T] -> [B, T, D]
    x = embed[x, :]
    sharding = jax.sharding.NamedSharding(mesh, jax.sharding.PartitionSpec('data', None, None))
    x = jax.lax.with_sharding_constraint(x, sharding)
    # [B, T, D] -> [B, T, F] -> [B, T, D]
    x = jnp.dot(jax.nn.relu(jnp.dot(x, w1)), w2)

    # [B, T, D] -> [B, T, V]
    return jax.nn.log_softmax(jnp.dot(x, embed.T))

import jax.sharding as shd
# Update this part of your code:
mesh = jax.make_mesh(
    axis_shapes=(8, 1), 
    axis_names=('data', 'model'),
    # Changing these to Auto allows JAX to reshard for you
    axis_types=(shd.AxisType.Auto, shd.AxisType.Auto) 
)
jax.sharding.set_mesh(mesh)

In [10]:
bs = 8
seqlen = 1024
vocab_size = 32_768
d_model = 8_192
d_ff = 16_384

tokens = jnp.zeros((bs, seqlen), dtype=jnp.int32)  # tokens of shape (batch_size, seq_len)
embed = jnp.zeros((vocab_size, d_model), jnp.bfloat16)  # a single weight matrix of shape (d_model, 2 * d_model)
w1 = jnp.zeros((d_model, d_ff), jnp.bfloat16)  # a single weight matrix of shape (d_model, 2 * d_model)
w2 = jnp.zeros((d_ff, d_model), jnp.bfloat16)  # a single weight matrix of shape (d_model, 2 * d_model)

forward_fn = jax.jit(forward)

out = forward_fn(tokens, embed, w1, w2)  # this compiles and runs the program

In [11]:
with jax.profiler.trace("/kaggle/working/"):
  out = forward_fn(tokens, embed, w1, w2)
  _ = jax.block_until_ready(out)

In [12]:
def update_step(x, y, embed, w1, w2, lr):
  def loss_fn(x, y, embed, w1, w2):
    logits = forward_fn(x, embed, w1, w2)
    return -jnp.mean(jnp.take_along_axis(logits, y[:, :, None], axis=-1))

  grad = jax.grad(loss_fn, argnums=(2, 3, 4))(x, y, embed, w1, w2)
  return jax.tree.map(lambda w, grad: w - lr * grad, (embed, w1, w2), grad)


In [13]:
x = tokens
y = jnp.roll(tokens, axis=1, shift=1)

In [14]:
new_embed, new_w1, new_w2 = jax.jit(update_step)(x, y, embed, w1, w2, jnp.asarray(1e-3))

# Run on multiple TPUs

In [15]:
# # Create a mesh with a data dimension of size 8, model of 1.
# mesh = jax.make_mesh(axis_shapes=(8, 1), axis_names=('data', 'model'))
# jax.sharding.set_mesh(mesh)

In [16]:
tokens = jax.device_put(tokens, P('data', None))  # sharded along the batch dimension
embed, w1, w2 = jax.tree.map(lambda arr: jax.device_put(arr, None), (embed, w1, w2))  # None means "replicated"

In [17]:
tokens

Array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], dtype=int32)

In [18]:
with jax.profiler.trace("/kaggle/working/"):
  out = forward_fn(tokens, embed, w1, w2)
  _ = jax.block_until_ready(out)

In [19]:
# print(url)

# What if we want to do something more complicated?

In [20]:
def forward(x, embed, w1, w2):
    # Match the rank of x: [Batch, Seq, Dim]
    # Shard only on the 'data' axis (Batch)
    data_sharding = jax.sharding.NamedSharding(mesh, P('data', None, None))
    
    x = embed[x]
    x = jax.lax.with_sharding_constraint(x, data_sharding)

    # For FSDP, we usually keep weights sharded and activations sharded
    x = jnp.dot(x, w1)
    x = jax.nn.relu(x)
    x = jnp.dot(x, w2)
    
    # Final projection
    logits = jnp.dot(x, embed.T)
    return jax.nn.log_softmax(logits)


import jax.sharding as shd
# Update this part of your code:
mesh = jax.make_mesh(
    axis_shapes=(8, 1), 
    axis_names=('data', 'model'),
    # Changing these to Auto allows JAX to reshard for you
    axis_types=(shd.AxisType.Auto, shd.AxisType.Auto) 
)
jax.sharding.set_mesh(mesh)


# # Create a mesh with a data dimension of size 64, model of 1.
# mesh = jax.make_mesh(axis_shapes=(8, 1), axis_names=('data', 'model'))
# jax.sharding.set_mesh(mesh)

forward_fn = jax.jit(forward)

In [21]:
# What about FSDP?

tokens = jax.device_put(tokens, P('data', None))  # sharded along the batch dimension

w1 = jax.device_put(w1, P('data', None))
w2 = jax.device_put(w2, P(None, 'data'))
embed = jax.device_put(embed, P(None, None))

In [22]:
with jax.profiler.trace("/kaggle/working/"):
  out = forward_fn(tokens, embed, w1, w2)
  _ = jax.block_until_ready(out)

In [23]:
# print(url[0])

# Model parallelism

In [24]:
def forward(x, embed, w1, w2):
    # x: [B, T]
    # embed: [V, D]
    # w1: [D, F]
    # w2: [F, D]

    # Embed input tokens [B, T] -> [B, T, D]
    x = embed[x] # Result: [B, T, D]
    
    # You are sharding the LAST dimension (D) across 'model' axis
    # x: [B, T, D] -> Sharded as [None, None, 'model']
    sharding = jax.sharding.NamedSharding(mesh, P(None, None, 'model'))
    x = jax.lax.with_sharding_constraint(x, sharding)

    # [B, T, D] @ [D, F] -> [B, T, F]
    # PROBLEM: Both x and w1 are sharded on D. 
    # This triggers the windowed_dot_general All-Gather loop in HLO.
    x = jnp.dot(x, w1) 
    x = jax.nn.relu(x)
    
    # [B, T, F] @ [F, D] -> [B, T, D]
    x = jnp.dot(x, w2)
    x = jax.lax.with_sharding_constraint(x, sharding)
    
    # # [B, T, D] @ [D, V] -> [B, T, V]
    # # embed.T is [D, V]. Both x and embed.T are sharded/contracting on D.
    # logits = jnp.dot(x, embed.T)
    # logits = jax.nn.log_softmax(logits) # Result: [B, T, V]
    
    # # Note: Using '==' here is a comparison, not an assignment.
    # # Result: [B, T, V]
    # logits == jax.lax.with_sharding_constraint(logits, sharding) 
    return x


import jax.sharding as shd
# Update this part of your code:
mesh = jax.make_mesh(
    axis_shapes=(1, 8), 
    axis_names=('data', 'model'),
    # Changing these to Auto allows JAX to reshard for you
    axis_types=(shd.AxisType.Auto, shd.AxisType.Auto) 
)
jax.sharding.set_mesh(mesh)


# # Create a mesh with a data dimension of size 64, model of 1.
# mesh = jax.make_mesh(axis_shapes=(8, 1), axis_names=('data', 'model'))
# jax.sharding.set_mesh(mesh)

forward_fn = jax.jit(forward)

In [25]:
mesh = jax.make_mesh(
    axis_shapes=(1, 8), 
    axis_names=('data', 'model'),
    # Changing these to Auto allows JAX to reshard for you
    axis_types=(shd.AxisType.Auto, shd.AxisType.Auto) 
)
jax.sharding.set_mesh(mesh)


# # Create a mesh with a data dimension of size 64, model of 1.
# mesh = jax.make_mesh(axis_shapes=(8, 8), axis_names=('data', 'model'))
# jax.sharding.set_mesh(mesh)

In [26]:
tokens = jax.device_put(tokens, P('data', None))  # sharded along the batch dimension

w1 = jax.device_put(w1, P(None, 'model'))
w2 = jax.device_put(w2, P('model', None))
embed = jax.device_put(embed, P(None, 'model'))

In [27]:
with jax.profiler.trace("/kaggle/working/"):
  out = forward_fn(tokens, embed, w1, w2)
  _ = jax.block_until_ready(out)

In [28]:
# print(url[0])

# Writing a "clean" LLM in JAX

In [29]:
from flax import struct
from typing import Callable

@struct.dataclass
class TensorInfo:
  shape: jax.ShapeDtypeStruct
  logical_axes: tuple[str, ...]
  initializer: Callable | None = None

@struct.dataclass
class Config:
  num_layers: int
  d_model: int
  d_ff: int
  vocab_size: int

  dtype: jnp.dtype = jnp.bfloat16


@struct.dataclass
class Layer:

  w1: jax.Array | TensorInfo
  w2: jax.Array | TensorInfo

  @classmethod
  def abstract(cls, cfg: Config):
    return Layer(
        w1=TensorInfo(
            jax.ShapeDtypeStruct((cfg.d_model, cfg.d_ff), cfg.dtype),
            ('d_model', 'd_ff'),
            jax.nn.initializers.he_normal(in_axis=0, out_axis=1)
        ),
        w2=TensorInfo(
            jax.ShapeDtypeStruct((cfg.d_ff, cfg.d_model), cfg.dtype),
            ('d_ff', 'd_model'),
            jax.nn.initializers.he_normal(in_axis=1, out_axis=0)
        ))


@struct.dataclass
class Weights:

  embedding: jax.Array | TensorInfo
  layers: list[Layer]

  @classmethod
  def abstract(cls, cfg: Config):
      return Weights(
        layers=[Layer.abstract(cfg) for _ in range(cfg.num_layers)],
        embedding=TensorInfo(
            jax.ShapeDtypeStruct((cfg.vocab_size, cfg.d_model), cfg.dtype),
            ('vocab', 'd_model'),
            jax.nn.initializers.he_normal(in_axis=0, out_axis=1)
        ))

  @classmethod
  def shardings(cls, cfg: Config, mesh: jax.sharding.Mesh, rules: dict):
      abstract = cls.abstract(cfg)
      return jax.tree.map(lambda info: _logical_to_sharding(info.logical_axes, mesh, rules), abstract, is_leaf=lambda x: isinstance(x, TensorInfo))

  @classmethod
  def init(cls, cfg: Config, key: jax.random.PRNGKey, mesh: jax.sharding.Mesh, rules: dict):
      abstract = cls.abstract(cfg)
      num_leaves = len(jax.tree_util.tree_leaves(abstract))
      key_iter = iter(jax.random.split(key, num_leaves))

      # 1. Force the generation to happen on the CPU Host
      with jax.default_device(jax.devices('cpu')[0]):
          cpu_weights = jax.tree.map(
              lambda info: info.initializer(next(key_iter), info.shape.shape, info.shape.dtype), 
              abstract, 
              is_leaf=lambda x: isinstance(x, TensorInfo)
          )

        # 2. Get the target physical shardings
      target_shardings = cls.shardings(cfg, mesh, rules)

        # 3. Beam the weights from CPU -> TPU natively into their sharded layout
      return jax.device_put(cpu_weights, target_shardings)


In [30]:
import collections

from typing import Tuple

ShardingRules = collections.namedtuple('ShardingRules',
 ['batch', 'sequence', 'd_model', 'd_ff', 'vocab'])

def _logical_to_physical(logical: Tuple[str, ...], rules: ShardingRules):
    """Converts logical to physical pspec."""
    return P(*(getattr(rules, l) if l is not None else None for l in logical ))

def _logical_to_sharding(logical: Tuple[str, ...], mesh: jax.sharding.Mesh, rules: ShardingRules):
    """Converts logical to sharding."""
    return jax.sharding.NamedSharding(mesh, _logical_to_physical(logical, rules))

fsdp_rules = ShardingRules(
    batch='x',
    sequence=None,
    d_model='x',
    d_ff=None,
    vocab=None
)

mdl_parallel_rules = ShardingRules(
    batch=None,
    sequence=None,
    d_model=None,
    d_ff='x',
    vocab=None
)

def _logical_to_physical(logical: P, rules: ShardingRules):
    """Converts logical to physical pspec."""
    return P(*(getattr(rules, l) for l in logical))

def _logical_to_sharding(logical: P, mesh: jax.sharding.Mesh, rules: ShardingRules):
    """Converts logical to sharding."""
    return jax.sharding.NamedSharding(mesh, _logical_to_physical(logical, rules))


In [31]:
mesh = jax.make_mesh((8,1), ('x', 'y'))
jax.sharding.set_mesh(mesh)

In [32]:
import gc
jax.clear_caches()
gc.collect()
cfg = Config(
    d_model=8192,
    d_ff=8192 * 4,
    num_layers=4,
    vocab_size=8192,
    dtype=jnp.bfloat16,
)

# mesh = jax.make_mesh((4,2), ('x', 'y'))

rules = fsdp_rules

rng = jax.random.PRNGKey(42)
weights = Weights.init(cfg, rng, mesh, rules)

# weights

In [33]:
def forward(x: jax.Array, weights: Weights):
  # 1. Embedding lookup (Fixed)
  out_spec = P('x', None, None) 
  out_sharding = jax.sharding.NamedSharding(mesh, out_spec)
  x = weights.embedding.at[x, :].get(out_sharding=out_sharding)

  for layer in weights.layers:
    # 2. Fix the Ambiguity:
    # We tell JAX: "I know both are sharded on 'y', please give me an 
    # output that is sharded on 'x' (Batch)." 
    # JAX will automatically insert the necessary All-Reduce.
    x = jnp.dot(x, layer.w1)
    x = jax.nn.relu(x)
    
    x = jnp.dot(x, layer.w2,out_sharding=out_sharding)
    x = jax.nn.relu(x)

  # 3. Final Projection
  logits = jnp.dot(x, weights.embedding.T, out_sharding=out_sharding)
  return jax.nn.log_softmax(logits)

# def forward(x: jax.Array, weights: Weights):

#   # Embed input tokens [B, T] -> [B, T, D]
#   x = weights.embedding[x, :]

#   for layer in weights.layers:
#     # [B, T, D] -> [B, T, D] -> [B, T, D]
#     x = jax.nn.relu(jnp.dot(x, layer.w1))
#     x = jax.nn.relu(jnp.dot(x, layer.w2))

#   # [B, T, D] -> [B, T, V]
#   return jax.nn.log_softmax(jnp.dot(x, weights.embedding.T))

In [34]:
batch_size = 32
seq_len = 1024

input_sharding = _logical_to_sharding(('batch', 'sequence'), mesh=mesh, rules=rules)
x = jnp.zeros((batch_size, seq_len), dtype=jnp.int32, device=input_sharding)

output_sharding = _logical_to_sharding(('batch', 'sequence', 'vocab'), mesh=mesh, rules=rules)
forward_fn = jax.jit(forward, out_shardings=output_sharding)

compiled = forward_fn.lower(x, weights).compile()

In [35]:
out = compiled(x, weights)
jax.block_until_ready(out)

# 2. PROFILING
# Now that the TPU is warm and the instructions are loaded, we capture the trace.
with jax.profiler.trace("/kaggle/working/"):
    # Pass the exact arguments you used during the .lower() step
    out = compiled(x, weights)
    
    # This forces the CPU to wait for the TPU to finish its asynchronous math
    jax.block_until_ready(out)

print("Trace complete! Download the .gz file from /kaggle/working/ and open in https://ui.perfetto.dev/")

Trace complete! Download the .gz file from /kaggle/working/ and open in https://ui.perfetto.dev/


# TLDR

In [36]:
# # Data parallelism
# activations = jax.device_put(activations, P('data', None, None))  # sharded along the batch dimension
# weights = jax.device_put(weights, P(None, None))  # unsharded

# # FSDP parallelism
# activations = jax.device_put(activations, P('data', None, None))  # sharded along the batch dimension
# weights = jax.device_put(weights, P('data', None))  # weights sharded and gathered just-in-time

# # Tensor parallelism
# activations = jax.device_put(activations, P('data', None, 'model'))  # sharded along the batch and hidden
# weights = jax.device_put(weights, P(None, 'model'))  # weights sharded and gathered just-in-time

# # Mixed FSDP + model parallelism
# activations = jax.device_put(activations, P('data', None, 'model'))  # sharded along the batch and hidden
# weights = jax.device_put(weights, P('data', 'model'))  # we do both! fully sharded weights

# Sharding in Types

In [37]:
from jax.sharding import PartitionSpec as P, AxisType, set_mesh, get_abstract_mesh
from jax.sharding import reshard, auto_axes, explicit_axes

In [38]:
# Create a mesh with a data dimension of size 64, model of 1.
mesh = jax.make_mesh(axis_shapes=(4, 2), axis_names=('data', 'model'),
                     axis_types=(AxisType.Explicit, AxisType.Explicit))
jax.sharding.set_mesh(mesh)

In [39]:
tokens = jnp.zeros((128, 1024), dtype=jnp.bfloat16)  # tokens of shape (batch_size, seq_len)
weights = jnp.zeros((1024, 2048), jnp.bfloat16)  # a single weight matrix of shape (d_model, 2 * d_model)

# data parallelism
tokens = jax.device_put(tokens, P('data', None))  # sharded along the batch dimension
weights = jax.device_put(weights, P('data', 'model'))  # unsharded

def update_step(tokens, weights):
  print(jax.typeof(tokens).sharding)
  out = jnp.matmul(tokens, weights)
  print(jax.typeof(out).sharding)
  return out

update_fn = jax.jit(update_step)  # some training step function, using jax.grad

new_weights = update_fn(tokens, weights)  # this compiles and runs the program

NamedSharding(mesh=AbstractMesh('data': 4, 'model': 2, axis_types=(Explicit, Explicit), device_kind=TPU v5 lite, num_cores=1), spec=P('data', None))
NamedSharding(mesh=AbstractMesh('data': 4, 'model': 2, axis_types=(Explicit, Explicit), device_kind=TPU v5 lite, num_cores=1), spec=P('data', 'model'))


# Shard Map

In [40]:
mesh = jax.make_mesh(axis_shapes=(4, 2), axis_names=('X', 'Y'))
jax.sharding.set_mesh(mesh)
import functools

import jax
import jax.numpy as jnp
import jax.sharding as shd
import numpy as np

from jax.experimental.shard_map import shard_map

mesh = jax.make_mesh(axis_shapes=(4, 2), axis_names=('X', 'Y'))
def P(*args):
  return shd.NamedSharding(mesh, shd.PartitionSpec(*args))

B, D, F = 1024, 2048, 8192
A = jnp.arange(np.prod((B, D))).reshape((B, D))
W = jnp.arange(np.prod((D, F))).reshape((D, F))

A = jax.device_put(A, P('X', 'Y'))
W = jax.device_put(W, P(None, 'Y'))

@functools.partial(jax.jit, out_shardings=P('X', 'Y'))
def matmul(lhs, rhs):
  return lhs @ rhs

def collective_matmul_allgather_lhs_contracting(lhs, rhs):
    axis_size = jax.lax.psum(1, axis_name='Y')  
    idx = jax.lax.axis_index('Y')

    chunk_size = lhs.shape[1]
    assert rhs.shape[0] % chunk_size == 0

    def f(i, carrys):
        accum, lhs = carrys
        rhs_chunk = jax.lax.dynamic_slice_in_dim(rhs, (idx + i) % axis_size * chunk_size, chunk_size)
        
        # Matmul for a chunk
        update = lhs @ rhs_chunk
        
        # Circular shift to the left
        lhs = jax.lax.ppermute(
            lhs,
            axis_name='Y',
            perm=[(j, (j - 1) % axis_size) for j in range(axis_size)]
        )
        return accum + update, lhs

    accum = jnp.zeros((lhs.shape[0], rhs.shape[1]), dtype=lhs.dtype)
    
    # --- THE FIX ---
    # Cast the newly created zeros array so JAX treats it as 'varying' across the mesh
    accum = jax.lax.pcast(accum, ('X', 'Y'), to='varying')
    # ---------------

    accum, lhs = jax.lax.fori_loop(0, axis_size - 1, f, (accum, lhs), unroll=True)

    # Compute the last chunk 
    i = axis_size - 1
    rhs_chunk = jax.lax.dynamic_slice_in_dim(rhs, (idx + i) % axis_size * chunk_size, chunk_size)
    update = lhs @ rhs_chunk
    
    return accum + update

jit_sharded_f = jax.jit(shard_map(
  collective_matmul_allgather_lhs_contracting, mesh,
  in_specs=(shd.PartitionSpec('X', 'Y'), shd.PartitionSpec(None, 'Y')), out_specs=shd.PartitionSpec('X', 'Y')))

shmapped_out = jit_sharded_f(A, W)
expected_out = matmul(A, W)

np.testing.assert_array_equal(shmapped_out, expected_out)

/tmp/ipykernel_12/1433663601.py:10: DeprecationWarning: jax.experimental.shard_map is deprecated in v0.8.0. Used jax.shard_map instead.
  from jax.experimental.shard_map import shard_map


## Mini Transformer

In [41]:
from flax import struct
from typing import Callable, Tuple

import collections

import jax
import jax.numpy as jnp

P = jax.sharding.PartitionSpec

ShardingRules = collections.namedtuple('ShardingRules',
 ['batch', 'sequence', 'd_model', 'query_heads', 'key_heads', 'key_dim', 'd_ff', 'vocab'])

def _logical_to_physical(logical: Tuple[str, ...], rules: ShardingRules):
    """Converts logical to physical pspec."""
    return P(*(getattr(rules, l) for l in logical))

def _logical_to_sharding(logical: Tuple[str, ...], mesh: jax.sharding.Mesh, rules: ShardingRules):
    """Converts logical to sharding."""
    return jax.sharding.NamedSharding(mesh, _logical_to_physical(logical, rules))


@struct.dataclass
class Config:
  d_model: int
  ffw_multiplier: int
  num_layers: int

  query_heads: int
  kv_heads: int
  key_dim: int

  vocab_size: int

  dtype: jnp.dtype = jnp.bfloat16


@struct.dataclass
class TensorInfo:
  shape: jax.ShapeDtypeStruct
  logical_axes: tuple[str, ...]
  initializer: Callable | None = None


@struct.dataclass
class Layer:

  q: jax.Array | TensorInfo
  k: jax.Array | TensorInfo
  v: jax.Array | TensorInfo
  proj: jax.Array | TensorInfo

  w1: jax.Array | TensorInfo
  w2: jax.Array | TensorInfo

  gamma1: jax.Array | TensorInfo
  gamma2: jax.Array | TensorInfo

  @classmethod
  def abstract(cls, cfg: Config):
    return Layer(
        q=TensorInfo(
            jax.ShapeDtypeStruct((cfg.d_model, cfg.query_heads, cfg.key_dim), dtype=cfg.dtype),
            ('d_model', 'query_heads', 'key_dim'),
            jax.nn.initializers.he_normal(in_axis=0, out_axis=(1,2)),
        ),
        k=TensorInfo(
            jax.ShapeDtypeStruct((cfg.d_model, cfg.kv_heads, cfg.key_dim), dtype=cfg.dtype),
            ('d_model', 'query_heads', 'key_dim'),
            jax.nn.initializers.he_normal(in_axis=0, out_axis=(1,2)),
        ),
        v=TensorInfo(
            jax.ShapeDtypeStruct((cfg.d_model, cfg.kv_heads, cfg.key_dim), dtype=cfg.dtype),
            ('d_model', 'query_heads', 'key_dim'),
            jax.nn.initializers.he_normal(in_axis=0, out_axis=(1,2)),
        ),
        proj=TensorInfo(
            jax.ShapeDtypeStruct((cfg.query_heads, cfg.key_dim, cfg.d_model), dtype=cfg.dtype),
            ('query_heads', 'key_dim', 'd_model'),
            jax.nn.initializers.he_normal(in_axis=(0,1), out_axis=2),
        ),
        w1=TensorInfo(
            jax.ShapeDtypeStruct((cfg.d_model, cfg.d_model * cfg.ffw_multiplier), dtype=cfg.dtype),
            ('d_model', 'd_ff'),
            jax.nn.initializers.he_normal(in_axis=0, out_axis=1),
        ),
        w2=TensorInfo(
            jax.ShapeDtypeStruct((cfg.d_model * cfg.ffw_multiplier, cfg.d_model), dtype=cfg.dtype),
            ('d_ff', 'd_model'),
            jax.nn.initializers.he_normal(in_axis=0, out_axis=1),
        ),
        gamma1=TensorInfo(
            jax.ShapeDtypeStruct((cfg.d_model,), dtype=cfg.dtype),
            ('d_model',),
            jax.nn.initializers.constant(1.0),
        ),
        gamma2=TensorInfo(
            jax.ShapeDtypeStruct((cfg.d_model,), dtype=cfg.dtype),
            ('d_model',),
            jax.nn.initializers.constant(1.0),
        ),
    )

@struct.dataclass
class Weights:
  layers: list[Layer]
  embedding: jax.Array | TensorInfo

  @classmethod
  def abstract(cls, cfg: Config):
      return Weights(
        layers=[Layer.abstract(cfg) for _ in range(cfg.num_layers)],
        embedding=TensorInfo(
          jax.ShapeDtypeStruct((cfg.vocab_size, cfg.d_model), cfg.dtype),
          ('vocab', 'd_ff'),
          jax.nn.initializers.he_normal(in_axis=0, out_axis=1)
        ),
    )

  @classmethod
  def sharding(cls, cfg: Config, mesh: jax.sharding.Mesh, rules: ShardingRules):
    abstract = cls.abstract(cfg)
    return jax.tree.map(lambda info: _logical_to_sharding(info.logical_axes, mesh, rules), abstract,
                        is_leaf=lambda x: isinstance(x, TensorInfo))

  @classmethod
  def init(cls, cfg: Config, key: jax.random.PRNGKey, mesh: jax.sharding.Mesh, rules: ShardingRules):
    abstract = cls.abstract(cfg)
    sharding = cls.sharding(cfg, mesh, rules)
    cpu_device = jax.devices('cpu')[0]
    cpu_key = jax.device_put(key, cpu_device)
    # 1. Safely generate a tree of PRNG keys that perfectly matches the abstract structure
    leaves, treedef = jax.tree_util.tree_flatten(abstract, is_leaf=lambda x: isinstance(x, TensorInfo))
    keys = jax.random.split(key, len(leaves))
    key_tree = jax.tree_util.tree_unflatten(treedef, keys)

    # 2. Force the generation to happen on the Host CPU to avoid TPU memory exhaustion
    with jax.default_device(jax.devices('cpu')[0]):
        def _init_leaf(info, k):
            return info.initializer(k, info.shape.shape, info.shape.dtype)

        # Generate all weights eagerly in system RAM
        cpu_weights = jax.tree_util.tree_map(
            _init_leaf,
            abstract,
            key_tree,
            is_leaf=lambda x: isinstance(x, TensorInfo)
        )

    # 3. Beam the weights directly from CPU to the TPU cores in their correct sharded layout
    return jax.device_put(cpu_weights, sharding)


In [42]:
def rms_norm(x: jax.Array, gamma: jax.Array) -> jax.Array:
    """Apply RMS normalization."""
    rms = jax.lax.rsqrt(jnp.mean(jnp.square(x), axis=-1, keepdims=True) + 1e-6)
    return gamma * x * rms

def layer_forward(x: jax.Array, layer: Layer) -> jax.Array:
  # NOTE: 'x' coming in MUST now be sharded P('x', None, 'y')
  with jax.named_scope('pre_attn_norm'):
    # rms_norm happens on the sharded activation, saving compute!
    attn_in = rms_norm(x, layer.gamma1)

  # Q, K, V projections
  with jax.named_scope('qkv_matmul'):
    # THE IMPLICIT ALL-GATHER: 
    # Because layer.q expects the full d_model, XLA will automatically insert 
    # the AllGatherY(In[BX, DY]) right here before doing the local matmuls!
    q = jnp.einsum('btd,dhq->bhtq', attn_in, layer.q)
    k = jnp.einsum('btd,dhk->bhtk', attn_in, layer.k)
    v = jnp.einsum('btd,dhv->bhtv', attn_in, layer.v)
  # TODO: add RoPE here

  # Attention
  with jax.named_scope('attn'):
    scale = q.shape[-1] ** -0.5
    # TODO: add masking here

    num_query_heads, num_kv_heads = q.shape[1], k.shape[1]

    if num_query_heads == num_kv_heads or num_kv_heads == 1:
      qk = jnp.einsum('bhtd,bhsd->bhts', q, k) * scale
      logits = jax.nn.softmax(qk.astype(jnp.float32), axis=-1)
      attn_vec = jnp.einsum('bhsd,bhts->bhtd', v, logits)
    else:
      assert num_query_heads % num_kv_heads == 0
      q = q.reshape(q.shape[0:1] +
                    (num_kv_heads, num_query_heads // num_kv_heads) +
                    q.shape[2:])
      qk = jnp.einsum('bqhtd,bhsd->bqhts', q, k) * scale
      logits = jax.nn.softmax(qk.astype(jnp.float32), axis=-1)
      attn_vec = jnp.einsum('bqsd,bqhts->bqhtd', v, logits)
      attn_vec = attn_vec.reshape(attn_vec.shape[0:1] + (num_query_heads,) + attn_vec.shape[3:])

  with jax.named_scope('attn_proj'):
    # THE REDUCE-SCATTER:
    # We tell XLA we want the output d_model to be sharded on 'y'.
    # It will sum the partials and scatter them across the chips.
    attn_out = jnp.einsum('bhtv,hvd->btd', attn_vec, layer.proj, out_sharding=P('x', None, 'y'))

    # Residual connection
  with jax.named_scope('residual'):
    # x is P('x', None, 'y') and attn_out is P('x', None, 'y'). Perfect match.
    x = x + attn_out

  # Second RMSNorm (Pre-LN for FFN)
  with jax.named_scope('ffn_pre_norm'):
    ffw_in = rms_norm(x, layer.gamma2)

  with jax.named_scope('ffw'):
    # ffw_in is P('x', None, 'y'). XLA inserts another AllGatherY here automatically.
    ffw_out = jnp.einsum('btd,df->btf', ffw_in, layer.w1).astype(jnp.bfloat16)
    ffw_out = jax.nn.gelu(ffw_out)
    
    # THE REDUCE-SCATTER:
    ffw_out = jnp.einsum('btf,fd->btd', ffw_out, layer.w2, out_sharding=P('x', None, 'y')).astype(jnp.bfloat16)

  # Residual connection
  with jax.named_scope('residual'):
    x = x + ffw_out

  return x


def forward(x: jax.Array, weights: Weights) -> jax.Array:
    """Forward pass through the network."""
    
    # Since d_model is None, the output shape [batch, seq, d_model] 
    # should simply be P('x', None, None).
    x = weights.embedding.at[x].get(out_sharding=P('x', None, None))
    
    for idx, layer in enumerate(weights.layers):
        with jax.named_scope(f'layer_{idx}'):
            x = layer_forward(x, layer)
    
    logits = jnp.einsum('vd,btd->btv', weights.embedding, x, out_sharding=P('x', None, None))
    return jax.nn.log_softmax(logits, axis=-1)


In [43]:
fsdp_rules = ShardingRules(
    batch=('x', 'y'),
    sequence=None,
    d_model=('x', 'y'),
    query_heads=None,
    key_heads=None,
    key_dim=None,
    d_ff=None,
    vocab=None
)

model_parallel_rules = ShardingRules(
    batch=None,
    sequence=None,
    d_model=None,
    query_heads=('x', 'y'),
    key_heads=('x', 'y'),
    key_dim=None,
    d_ff=('x', 'y'),
    vocab=None
)

mixed_rules = ShardingRules(
    batch='x',
    sequence=None,
    d_model=None,    # <-- FIX: Leave replicated so it doesn't collide with batch or heads
    query_heads='y',
    key_heads='y',
    key_dim=None,
    d_ff='y',
    vocab=None       # Pro-tip: Change this to 'y' later to shard your massive embedding table!
)

In [44]:
for arr in jax.live_arrays():
  arr.delete()

In [45]:
jax.clear_caches()
cfg = Config(
    d_model=8192,
    ffw_multiplier=4,
    num_layers=8,
    query_heads=16,
    kv_heads=16,
    key_dim=256,
    vocab_size=32_128,
    dtype=jnp.bfloat16,
)

mesh = jax.make_mesh((4, 2), ('x', 'y'))
jax.sharding.set_mesh(mesh)

rules = mixed_rules

rng = jax.random.PRNGKey(42)
weights = Weights.init(cfg, rng, mesh, rules)

In [46]:
batch_size = 8
seq_len = 1024

input_sharding = _logical_to_sharding(('batch', 'sequence'), mesh=mesh, rules=rules)
x = jnp.zeros((batch_size, seq_len), dtype=jnp.int32, device=input_sharding)

compiled = jax.jit(forward).lower(x, weights).compile()

In [47]:
with jax.profiler.trace("/kaggle/working/"):
  logits = compiled(x, weights)
  jax.block_until_ready(logits)